# EfficientNet-B0 Vehicle Classification (Keras / TensorFlow)

This notebook trains and evaluates an **EfficientNet-B0** convolutional neural network on the filtered vehicle dataset (32 makes) derived from `resized_640x640`.

### Key Pipeline Highlights:
- **Framework**: TensorFlow 2.x & Keras
- **Dataset**: 567,242 images across 32 vehicle makes (Train: ~453k, Val: ~56k, Test: ~56k)
- **Input Pipeline**: Asynchronous GPU-prefetching `tf.data` pipeline
- **Hardware Acceleration**: Automatic GPU detection, memory growth, and Mixed Precision (`mixed_float16`)
- **Training Strategy**: 2-stage transfer learning (Warmup classification head -> Fine-tuning top convolutional layers)
- **Evaluation**: Top-1 Accuracy, Top-5 Accuracy, confusion metrics, and per-class classification report


### 1. Environment & Hardware Setup


In [ ]:
import os
import sys
import json
import time
from pathlib import Path

# Ensure Matplotlib and Keras caches use writable directories
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib_cache")
os.environ.setdefault("KERAS_HOME", "/home/researchadmin/Econ/.keras")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "1")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, mixed_precision
from tensorflow.keras.applications import EfficientNetB0

print(f"TensorFlow Version: {tf.__version__}")

# GPU Configuration
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    gpu_details = tf.config.experimental.get_device_details(gpus[0])
    gpu_name = gpu_details.get("device_name", "NVIDIA GPU")
    print(f"Active GPU        : {gpu_name}")
else:
    print("WARNING: No GPU detected, running on CPU.")

# Enable Mixed Precision (FP16 compute + FP32 weights) for fast Tensor Core training
policy = mixed_precision.Policy("mixed_float16")
mixed_precision.set_global_policy(policy)
print(f"Mixed Precision   : compute={policy.compute_dtype}, variable={policy.variable_dtype}")


### 2. Configuration & Hyperparameters


In [ ]:
# Paths
SPLITS_DIR = Path("/home/researchadmin/Econ/resized_640x640/splits_filtered")
OUTPUT_DIR = Path("/home/researchadmin/Econ/repo-clone/stanford-cars-model/code/current/output_efficientnet_b0")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Training Hyperparameters
IMG_SIZE         = 224        # Native EfficientNet-B0 resolution (can be set up to 640)
BATCH_SIZE       = 64         # Batch size (suitable for RTX 4090 24GB)
WARMUP_EPOCHS    = 2          # Phase 1: Train top classification head only
FINE_TUNE_EPOCHS = 15         # Phase 2: Fine-tune top layers of backbone
WARMUP_LR        = 1e-3       # Warmup learning rate
FINE_TUNE_LR     = 1e-4       # Fine-tuning learning rate
UNFREEZE_LAYERS  = 40         # Top N layers of EfficientNet to unfreeze during fine-tuning
DROPOUT_RATE     = 0.4        # Dropout before softmax output layer
RANDOM_SEED      = 42

tf.keras.utils.set_random_seed(RANDOM_SEED)

print(f"Configuration set: IMG_SIZE={IMG_SIZE}, BATCH_SIZE={BATCH_SIZE}")
print(f"Outputs will be saved to: {OUTPUT_DIR}")


### 3. Load Splits & Label Map


In [ ]:
train_csv = SPLITS_DIR / "train.csv"
val_csv   = SPLITS_DIR / "val.csv"
test_csv  = SPLITS_DIR / "test.csv"
label_map_file = SPLITS_DIR / "label_map.json"

for p in [train_csv, val_csv, test_csv, label_map_file]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")

with open(label_map_file, "r") as f:
    label_meta = json.load(f)

idx_to_class = {int(k): v for k, v in label_meta["idx_to_class"].items()}
class_to_idx = {k: int(v) for k, v in label_meta["class_to_idx"].items()}
NUM_CLASSES = len(idx_to_class)
CLASS_NAMES = [idx_to_class[i] for i in range(NUM_CLASSES)]

df_train = pd.read_csv(train_csv)
df_val   = pd.read_csv(val_csv)
df_test  = pd.read_csv(test_csv)

print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES[:8]} ...")
print(f"Train samples : {len(df_train):,}")
print(f"Val samples   : {len(df_val):,}")
print(f"Test samples  : {len(df_test):,}")

df_train.head(3)


### 4. High-Performance tf.data Input Pipeline

EfficientNet models have built-in normalization (`[0, 255] -> [-1, 1]`), so raw float pixel values in range [0, 255] are passed directly.
We apply random horizontal flipping and subtle brightness/contrast jittering for training data.


In [ ]:
def build_dataset(df, img_size=224, batch_size=64, is_training=False):
    paths = df["image_path"].values
    labels = df["label"].values.astype("int32")

    def parse_image(path, label):
        img_bytes = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img_bytes, channels=3)
        img = tf.image.resize(img, [img_size, img_size])
        if is_training:
            img = tf.image.random_flip_left_right(img)
            img = tf.image.random_brightness(img, max_delta=0.1)
            img = tf.image.random_contrast(img, lower=0.9, upper=1.1)
            img = tf.clip_by_value(img, 0.0, 255.0)
        return img, label

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if is_training:
        dataset = dataset.shuffle(buffer_size=10000, reshuffle_each_iteration=True)

    dataset = dataset.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

print("Creating tf.data pipelines...")
train_ds = build_dataset(df_train, img_size=IMG_SIZE, batch_size=BATCH_SIZE, is_training=True)
val_ds   = build_dataset(df_val,   img_size=IMG_SIZE, batch_size=BATCH_SIZE, is_training=False)
test_ds  = build_dataset(df_test,  img_size=IMG_SIZE, batch_size=BATCH_SIZE, is_training=False)
print("Data pipelines ready.")


### 5. Inspect Training Samples


In [ ]:
sample_images, sample_labels = next(iter(train_ds))

plt.figure(figsize=(14, 7))
for i in range(min(8, len(sample_images))):
    ax = plt.subplot(2, 4, i + 1)
    img_np = sample_images[i].numpy().astype("uint8")
    lbl_idx = sample_labels[i].numpy()
    plt.imshow(img_np)
    plt.title(f"{idx_to_class[lbl_idx]} ({lbl_idx})", fontsize=11, fontweight="bold")
    plt.axis("off")
plt.tight_layout()
plt.show()


### 6. Build EfficientNet-B0 Model

We initialize **EfficientNet-B0** with pre-trained ImageNet weights, attach a Global Average Pooling layer, Batch Normalization, Dropout, and a dense output layer with 32 units. Note that the output layer is explicitly forced to `float32` for numerical stability under mixed precision.


In [ ]:
def build_model(num_classes, img_size=224, dropout=0.4):
    inputs = layers.Input(shape=(img_size, img_size, 3), name="input_image")
    
    base_model = EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_tensor=inputs,
    )
    
    x = layers.GlobalAveragePooling2D(name="avg_pool")(base_model.output)
    x = layers.BatchNormalization(name="head_bn")(x)
    x = layers.Dropout(dropout, name="top_dropout")(x)
    outputs = layers.Dense(num_classes, activation="softmax", dtype="float32", name="predictions")(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="EfficientNetB0_VehicleClassifier")
    return model, base_model

model, base_model = build_model(NUM_CLASSES, img_size=IMG_SIZE, dropout=DROPOUT_RATE)
print(f"Total Base Model Layers: {len(base_model.layers)}")
model.summary(show_trainable=True)


### 7. Training Callbacks Configuration


In [ ]:
best_model_path = OUTPUT_DIR / "efficientnet_b0_best.keras"
final_model_path = OUTPUT_DIR / "efficientnet_b0_final.keras"
history_csv_path = OUTPUT_DIR / "training_history.csv"
tb_log_dir = OUTPUT_DIR / "tensorboard_logs"

cb_list = [
    callbacks.ModelCheckpoint(
        filepath=str(best_model_path),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=4,
        restore_best_weights=True,
        verbose=1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
    callbacks.CSVLogger(
        filename=str(history_csv_path),
        append=True,
    ),
    callbacks.TensorBoard(
        log_dir=str(tb_log_dir),
        histogram_freq=0,
        update_freq="epoch",
    ),
]

print("Callbacks configured:")
print(f"  - ModelCheckpoint: {best_model_path}")
print(f"  - CSVLogger:       {history_csv_path}")
print(f"  - TensorBoard:     {tb_log_dir}")


### 8. Phase 1: Classification Head Warmup

In this phase, we freeze the entire EfficientNet-B0 backbone and train only the newly initialized classification head for 2 epochs with learning rate `1e-3`.


In [ ]:
# Freeze base model
base_model.trainable = False
print("Base model frozen. Only classification head is trainable.")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=WARMUP_LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy", tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top_5_accuracy")],
)

print(f"Starting Warmup Training for {WARMUP_EPOCHS} epochs...")
warmup_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=WARMUP_EPOCHS,
    callbacks=cb_list,
    verbose=1,
)


### 9. Phase 2: Fine-Tuning Top Backbone Layers

Now, we unfreeze the top layers of the backbone and continue training with a reduced learning rate (`1e-4`) for deeper feature adaptation.


In [ ]:
# Unfreeze top layers of EfficientNet
base_model.trainable = True
for layer in base_model.layers[:-UNFREEZE_LAYERS]:
    layer.trainable = False
for layer in base_model.layers[-UNFREEZE_LAYERS:]:
    layer.trainable = True

print(f"Unfroze top {UNFREEZE_LAYERS} layers of EfficientNet-B0 ({len(base_model.layers) - UNFREEZE_LAYERS} frozen).")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=FINE_TUNE_LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy", tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top_5_accuracy")],
)

total_epochs = WARMUP_EPOCHS + FINE_TUNE_EPOCHS
print(f"Starting Fine-Tuning from epoch {WARMUP_EPOCHS + 1} to {total_epochs}...")

fine_tune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=WARMUP_EPOCHS,
    epochs=total_epochs,
    callbacks=cb_list,
    verbose=1,
)

# Save final model
model.save(str(final_model_path))
print(f"Final model saved to: {final_model_path}")


### 10. Training & Validation Curves


In [ ]:
history_df = pd.read_csv(history_csv_path)

plt.figure(figsize=(14, 5))

# Accuracy curve
plt.subplot(1, 2, 1)
plt.plot(history_df["epoch"] + 1, history_df["accuracy"], "o-", label="Train Accuracy", linewidth=2)
plt.plot(history_df["epoch"] + 1, history_df["val_accuracy"], "s--", label="Val Accuracy", linewidth=2)
plt.title("Model Accuracy Across Epochs", fontsize=13, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

# Loss curve
plt.subplot(1, 2, 2)
plt.plot(history_df["epoch"] + 1, history_df["loss"], "o-", label="Train Loss", linewidth=2, color="crimson")
plt.plot(history_df["epoch"] + 1, history_df["val_loss"], "s--", label="Val Loss", linewidth=2, color="darkorange")
plt.title("Model Loss Across Epochs", fontsize=13, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150)
plt.show()


### 11. Final Evaluation on Official Test Split


In [ ]:
print(f"Loading best checkpoint for evaluation: {best_model_path}")
best_model = models.load_model(str(best_model_path))

test_results = best_model.evaluate(test_ds, verbose=1)
metric_names = best_model.metrics_names
metrics_summary = dict(zip(metric_names, [float(v) for v in test_results]))

print("\n" + "="*50)
print("              TEST SET EVALUATION               ")
print("="*50)
for k, v in metrics_summary.items():
    print(f"  {k:<22}: {v:.4f}")
print("="*50)

# Predictions & Classification Report
print("\nGenerating predictions across all test samples...")
preds = best_model.predict(test_ds, verbose=1)
y_pred = np.argmax(preds, axis=-1)
y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)

report_dict = classification_report(
    y_true,
    y_pred,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv(OUTPUT_DIR / "test_classification_report.csv")

metrics_summary["macro_f1"] = float(report_dict["macro avg"]["f1-score"])
metrics_summary["weighted_f1"] = float(report_dict["weighted avg"]["f1-score"])

with open(OUTPUT_DIR / "test_metrics.json", "w") as f:
    json.dump(metrics_summary, f, indent=2)

print("\nClassification Report (Top 10 Classes):")
report_df.head(10)


### 12. Per-Class F1-Score Breakdown


In [ ]:
per_class_f1 = report_df.loc[CLASS_NAMES, "f1-score"].sort_values()

plt.figure(figsize=(12, 10))
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(per_class_f1)))
bars = plt.barh(per_class_f1.index, per_class_f1.values, color=colors, edgecolor="black", alpha=0.85)

for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.01, bar.get_y() + bar.get_height() / 2, f"{width:.2f}",
             ha="left", va="center", fontsize=9, fontweight="bold")

plt.xlim(0, 1.1)
plt.title("Per-Class F1-Score on Test Set (32 Makes)", fontsize=14, fontweight="bold")
plt.xlabel("F1-Score")
plt.ylabel("Vehicle Make")
plt.grid(True, linestyle="--", alpha=0.5, axis="x")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "per_class_f1_scores.png", dpi=150)
plt.show()


### 13. Visualizing Test Predictions


In [ ]:
test_iter = iter(test_ds)
sample_test_images, sample_test_labels = next(test_iter)
sample_preds = best_model.predict(sample_test_images, verbose=0)
sample_pred_labels = np.argmax(sample_preds, axis=-1)
sample_confidences = np.max(sample_preds, axis=-1)

plt.figure(figsize=(15, 8))
for i in range(min(8, len(sample_test_images))):
    ax = plt.subplot(2, 4, i + 1)
    img_np = sample_test_images[i].numpy().astype("uint8")
    true_label = idx_to_class[sample_test_labels[i].numpy()]
    pred_label = idx_to_class[sample_pred_labels[i]]
    conf = sample_confidences[i]
    
    is_correct = (true_label == pred_label)
    title_color = "green" if is_correct else "red"
    
    plt.imshow(img_np)
    plt.title(f"True: {true_label}\nPred: {pred_label} ({conf*100:.1f}%)",
              color=title_color, fontsize=10, fontweight="bold")
    plt.axis("off")

plt.tight_layout()
plt.show()
